In [ ]:
import numpy as np
import pandas as pd
import re
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [ ]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [ ]:
df=pd.read_csv("/content/WELFake_Dataset.csv.zip")

In [ ]:
df.shape

(72134, 4)

In [ ]:
df.head()

,Unnamed: 0,title,text,label
0,0,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,No comment is expected from Barack Obama Membe...,1
1,1,NaN,Did they post their votes for Hillary already?,1
2,2,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...,"Now, most of the demonstrators gathered last ...",1
3,3,"Bobby Jindal, raised Hindu, uses story of Chri...",A dozen politically active pastors came here f...,0
4,4,SATAN 2: Russia unvelis an image of its terrif...,"The RS-28 Sarmat missile, dubbed Satan 2, will...",1


In [ ]:
df.isnull().sum()

,0
Unnamed: 0,0
title,558
text,39
label,0


In [ ]:
df=df.fillna('')

In [ ]:
X=df.drop(columns='label',axis=1)
y=df['label']

In [ ]:
print(X)
print(y)

       Unnamed: 0  ...                                               text
0               0  ...  No comment is expected from Barack Obama Membe...
1               1  ...     Did they post their votes for Hillary already?
2               2  ...   Now, most of the demonstrators gathered last ...
3               3  ...  A dozen politically active pastors came here f...
4               4  ...  The RS-28 Sarmat missile, dubbed Satan 2, will...
...           ...  ...                                                ...
72129       72129  ...  WASHINGTON (Reuters) - Hackers believed to be ...
72130       72130  ...  You know, because in fantasyland Republicans n...
72131       72131  ...  Migrants Refuse To Leave Train At Refugee Camp...
72132       72132  ...  MEXICO CITY (Reuters) - Donald Trump’s combati...
72133       72133  ...  Goldman Sachs Endorses Hillary Clinton For Pre...

[72134 rows x 3 columns]
0        1
1        1
2        1
3        0
4        1
        ..
72129    0
72130    

In [ ]:
ps=PorterStemmer()

In [ ]:
def stemming(content):
  stemmed_content = re.sub('[^a-zA-Z]',' ',content)
  stemmed_content = stemmed_content.lower()
  stemmed_content = stemmed_content.split()
  stemmed_content = [ps.stem(word) for word in stemmed_content if not word in stopwords.words('english')]
  stemmed_content = ' '.join(stemmed_content)
  return stemmed_content


In [ ]:
df['title']=df['title'].apply(stemming)

In [ ]:
print(df['title'])

0        law enforc high alert follow threat cop white ...
1                                                         
2        unbeliev obama attorney gener say charlott rio...
3        bobbi jindal rais hindu use stori christian co...
4        satan russia unv imag terrifi new supernuk wes...
                               ...                        
72129    russian steal research trump hack u democrat p...
72130    watch giuliani demand democrat apolog trump ra...
72131         migrant refus leav train refuge camp hungari
72132    trump tussl give unpopular mexican leader much...
72133           goldman sach endors hillari clinton presid
Name: title, Length: 72134, dtype: object


In [ ]:
X=df['title'].values
y=df['label'].values

In [ ]:
vectorizer=TfidfVectorizer()
vectorizer.fit(X)
X=vectorizer.transform(X)

In [ ]:
X_train,X_test,Y_train,Y_test=train_test_split(X,y,test_size=0.2,stratify=y,random_state=2)

In [ ]:
model=LogisticRegression()


In [ ]:
model.fit(X_train,Y_train)

LogisticRegression()

In [ ]:
X_pred=model.predict(X_train)
accuracy=accuracy_score(X_pred,Y_train)
print(accuracy)

0.9193858630668723


In [ ]:
X_predtest=model.predict(X_test)
accuracy=accuracy_score(X_predtest,Y_test)
print(accuracy)

0.900603035974215


In [ ]:
X_new = X_test[3]

prediction = model.predict(X_new)
print(prediction)

if (prediction[0]==0):
  print('The news is Real')
else:
  print('The news is Fake')

[0]
The news is Real


In [ ]:
import pickle

pickle.dump(model, open('model.pkl','wb'))
pickle.dump(vectorizer, open('vectorizer.pkl','wb'))

In [ ]:
from google.colab import files
files.download('model.pkl')
files.download('vectorizer.pkl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
proba = model.predict_proba(X_new)

confidence = max(proba[0])

prediction = model.predict(X_new)

if confidence < 0.6:
    print("The input does not appear to be news content")
else:
    if prediction[0] == 0:
        print("News is Real")
    else:
        print("News is Fake")